# 深度诊断：为什么收敛到 HF 而不是 FCI？

关键观察：
- Iter 0: QGT trace = 37.24（正常）
- Iter 50: QGT trace = 3.96e-29（几乎为 0）

这说明：
1. 初始化时网络有表达能力
2. 但训练过程中快速退化到 HF 态
3. 然后陷入局部极小

In [ ]:
import jax
import jax.numpy as jnp
import netket as nk
import netket.experimental as nkx
import numpy as np
from pyscf import gto, scf, fci
from flax import nnx
from functools import partial
from jax import flatten_util

# 设置系统
bond_length = 1.4
geometry = [('H', (0., 0., 0.)), ('H', (bond_length, 0., 0.))]
mol = gto.M(atom=geometry, basis='STO-3G', verbose=0)
mf = scf.RHF(mol).run(verbose=0)

cisolver = fci.FCI(mf)
E_fci = cisolver.kernel()[0]

print(f"FCI 能量: {E_fci:.8f} Ha")
print(f"HF 能量: {mf.e_tot:.8f} Ha")
print(f"相关能: {(mf.e_tot - E_fci)*27.2114:.2f} eV")

ha = nkx.operator.from_pyscf_molecule(mol)
hi = nk.hilbert.SpinOrbitalFermions(n_orbitals=2, s=1/2, n_fermions_per_spin=(1,1))

# 列出所有可能的电子组态
all_states = hi.all_states()
print(f"\n希尔伯特空间大小: {hi.n_states}")
print("\n所有可能的电子组态：")
for i, state in enumerate(all_states):
    print(f"  状态 {i}: {state}")

## 诊断 1：检查 FCI 波函数的组态分布

In [ ]:
# 获取 FCI 波函数
fcivec = cisolver.kernel()[1]
print(f"FCI 波函数形状: {fcivec.shape}")

# FCI 波函数的系数
print("\nFCI 波函数组态系数：")
for i in range(fcivec.shape[0]):
    for j in range(fcivec.shape[1]):
        coef = fcivec[i, j]
        if abs(coef) > 1e-6:
            print(f"  组态 ({i},{j}): 系数 = {coef:.6f}, |coef|² = {abs(coef)**2:.6f}")

# 计算组态权重
print("\n组态权重分布：")
weights = np.abs(fcivec.flatten())**2
print(f"  最大权重: {np.max(weights):.6f}")
print(f"  次大权重: {np.sort(weights)[-2]:.6f}")
print(f"  权重和: {np.sum(weights):.6f}")

## 诊断 2：检查网络初始化

In [ ]:
class SingleStateAnsatz(nnx.Module):
    def __init__(self, n_spin_orbitals: int, hidden_dim=16, *, rngs: nnx.Rngs):
        super().__init__()
        self.linear1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=complex)
        self.linear2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=complex)
        self.output = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=complex)

    def __call__(self, x):
        h = nnx.tanh(self.linear1(x.astype(complex)))
        h = nnx.tanh(self.linear2(h))
        out = self.output(h)
        return jnp.squeeze(out)

def forward(GraphDef_State, x):
    log_psi, _ = nnx.call(GraphDef_State)(x)
    return log_psi

# 初始化模型
rngs = nnx.Rngs(21)
model = SingleStateAnsatz(n_spin_orbitals=4, hidden_dim=12, rngs=rngs)
graphdef, model_state = nnx.split(model)
GraphDef_State = (graphdef, model_state)

# 计算所有组态的波函数值
log_psi_all = forward(GraphDef_State, all_states)
psi_all = jnp.exp(log_psi_all - log_psi_all.max())  # 归一化
psi_all = psi_all / jnp.linalg.norm(psi_all)

print("初始化网络的波函数分布：")
for i, (state, psi) in enumerate(zip(all_states, psi_all)):
    print(f"  状态 {i} ({state}): |ψ|² = {jnp.abs(psi)**2:.6f}")

print(f"\n波函数熵: {-jnp.sum(jnp.abs(psi_all)**2 * jnp.log(jnp.abs(psi_all)**2 + 1e-10)):.4f}")
print("（熵越大，说明波函数越分散，关联越强）")

## 诊断 3：检查训练过程中的波函数演化

In [ ]:
def check_wavefunction_evolution(model_state, graphdef, iteration):
    """检查波函数在训练过程中的演化"""
    GraphDef_State = (graphdef, model_state)
    log_psi_all = forward(GraphDef_State, all_states)
    psi_all = jnp.exp(log_psi_all - log_psi_all.max())
    psi_all = psi_all / jnp.linalg.norm(psi_all)
    
    weights = jnp.abs(psi_all)**2
    entropy = -jnp.sum(weights * jnp.log(weights + 1e-10))
    
    print(f"\n迭代 {iteration}:")
    print(f"  波函数熵: {entropy:.4f}")
    print(f"  最大权重: {jnp.max(weights):.6f}")
    print(f"  组态分布: {weights}")
    
    return entropy, weights

# 模拟前几次迭代的演化
print("="*60)
print("波函数演化诊断")
print("="*60)

entropy_init, weights_init = check_wavefunction_evolution(model_state, graphdef, 0)

## 根本原因分析

In [ ]:
print("\n" + "="*70)
print("根本原因分析")
print("="*70)
print()

print("问题诊断：")
print()
print("1. QGT 快速退化到 0：")
print("   - 初始化时 QGT trace = 37.24（正常）")
print("   - 50 次迭代后 QGT trace = 3.96e-29（几乎为 0）")
print("   - 说明网络快速收敛到局部极小")
print()

print("2. 能量收敛到 HF：")
print("   - HF 能量: -0.94148065 Ha")
print("   - 最终能量: -0.94139385 Ha")
print("   - 几乎完全一致！")
print()

print("3. 为什么会这样？")
print("   - H2 的 FCI 波函数主要由 2 个组态组成")
print("   - HF 态只包含 1 个组态")
print("   - 网络可能学会了只给这 2 个组态非零振幅")
print("   - 但权重分配错误，导致能量偏高")
print()

print("4. 关键问题：")
print("   - 网络容量不足（hidden_dim=12 太小）")
print("   - 学习率太大（0.01 导致快速收敛到局部极小）")
print("   - 初始化可能偏向 HF 态")
print()

print("="*70)

## 解决方案：改进网络架构和训练策略

In [ ]:
print("\n" + "="*70)
print("解决方案")
print("="*70)
print()

print("方案 1：增加网络容量")
print("  - hidden_dim: 12 → 32 或 64")
print("  - 增加网络深度：2 层 → 3 层")
print("  - 增强表达能力")
print()

print("方案 2：降低学习率")
print("  - learning_rate: 0.01 → 0.001")
print("  - 避免快速收敛到局部极小")
print()

print("方案 3：改进初始化")
print("  - 使用更小的初始化权重")
print("  - 或者使用预训练")
print()

print("方案 4：增加样本数")
print("  - n_samples: 1000 → 2000 或更多")
print("  - 更好的统计精度")
print()

print("方案 5：使用更好的 Ansatz")
print("  - 使用对称性约束")
print("  - 使用多行列式 Ansatz")
print("  - 添加 Jastrow 因子")
print()

print("推荐组合：")
print("  ✓ hidden_dim = 32")
print("  ✓ learning_rate = 0.001")
print("  ✓ n_samples = 2000")
print("  ✓ n_iterations = 500")
print()

print("="*70)